## 12b - Final Test Evaluation

This notebook performs the final test evaluation of the models selected during the validation stage:

- Demographics + Questionnaire - Logistic Regression
- Wearable + Questionnaire - Random Forest
- Full Multimodal - Random Forest 

The saved fitted pipelines are loaded directly and applied to the test data without refitting or retuning.

#### Objective

The objective of this notebook is to assess the final generalization performance of the validation-selected models on unseen test data and compare their test performance with the results obtained during validation.

The notebook will:

- Load the models selected during validation.
- Evaluate each selected model on its corresponding held-out test dataset.
- Calculate final performance metrics, including Accuracy, Balanced Accuracy, Macro F1, Macro - - Precision, and Macro Recall.
- Generate final confusion matrices.
- Compare validation and test performance for each selected model.

#### 1. Setup

In [1]:
from pathlib import Path
import os
import sys
import re
import warnings

os.environ.pop("MPLBACKEND", None)

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# ==========================================================
# Locate project root
# ==========================================================

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

# ========================================================
# Import shared evaluation utilities
# ========================================================

from src.modeling.evaluation import (
    calculate_metrics,
    get_classification_report,
    get_confusion_matrix,
)

# =======================================================
# Project directories
# =======================================================

DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

PROCESSED_DIR = (
    DATA_DIR
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

OUTPUTS_DIR = (
    PROJECT_ROOT
    / "outputs"
)

METRICS_DIR = (
    OUTPUTS_DIR
    / "metrics"
)

TABLES_DIR = (
    OUTPUTS_DIR
    / "tables"
)

FIGURES_DIR = (
    OUTPUTS_DIR
    / "figures"
)

for directory in [
    METRICS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# =======================================================
# Shared columns
# =======================================================

TARGET = "label"
ID_COLUMN = "patient_id"

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Processed data:", PROCESSED_DIR)
print("Metrics directory:", METRICS_DIR)
print("Tables directory:", TABLES_DIR)

KeyboardInterrupt: 

#### 2. Load Validation-Selected Models

In [ ]:
VALIDATION_BEST_MODELS_FILE = (
    TABLES_DIR
    / "validation_best_models.csv"
)

VALIDATION_RESULTS_FILE = (
    METRICS_DIR
    / "validation_results.csv"
)

if not VALIDATION_BEST_MODELS_FILE.exists():
    raise FileNotFoundError(
        f"Missing validation-selected model file: "
        f"{VALIDATION_BEST_MODELS_FILE}"
    )

if not VALIDATION_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Missing validation results file: "
        f"{VALIDATION_RESULTS_FILE}"
    )

selected_candidates = pd.read_csv(
    VALIDATION_BEST_MODELS_FILE
)

validation_results = pd.read_csv(
    VALIDATION_RESULTS_FILE
)

print(
    "Validation-selected candidates:",
    len(selected_candidates)
)

display(
    selected_candidates[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

#### 3. Define and Verify Final Test Inputs

In [ ]:
MODEL_FILES = {
    (
        "Demographics + Questionnaire",
        "Logistic Regression",
    ):
        MODELS_DIR
        / "demographics_plus_questionnaire_logistic_regression.joblib",

    (
        "Wearable + Questionnaire",
        "Random Forest",
    ):
        MODELS_DIR
        / "wearable_plus_questionnaire_random_forest.joblib",

    (
        "Full Multimodal",
        "Random Forest",
    ):
        MODELS_DIR
        / "full_multimodal_random_forest.joblib",
}

In [ ]:
# =============================================================================
# Define held-out test datasets
# =============================================================================

TEST_FILES = {
    "Demographics + Questionnaire":
        PROCESSED_DIR
        / "test_demographics_questionnaire_task_aware.csv",

    "Wearable + Questionnaire":
        PROCESSED_DIR
        / "test_wearable_questionnaire_task_aware.csv",

    "Full Multimodal":
        PROCESSED_DIR
        / "test_multimodal_full_task_aware.csv",
}

In [ ]:
# =============================================================================
# Verify required inputs
# =============================================================================

input_check_rows = []

for _, candidate in selected_candidates.iterrows():

    dataset = candidate["dataset"]
    model_name = candidate["model"]

    model_path = MODEL_FILES.get(
        (
            dataset,
            model_name,
        )
    )

    test_path = TEST_FILES.get(
        dataset
    )

    input_check_rows.append(
        {
            "Dataset": dataset,
            "Model": model_name,

            "Model File":
                model_path.name
                if model_path is not None
                else None,

            "Model Available":
                model_path.exists()
                if model_path is not None
                else False,

            "Test File":
                test_path.name
                if test_path is not None
                else None,

            "Test Available":
                test_path.exists()
                if test_path is not None
                else False,
        }
    )

input_check = pd.DataFrame(
    input_check_rows
)

display(input_check)

In [ ]:
assert input_check[
    "Model Available"
].all(), (
    "One or more selected model artifacts are missing."
)

assert input_check[
    "Test Available"
].all(), (
    "One or more held-out test datasets are missing."
)

print(
    "All final test inputs verified"
)

#### 4. Verify Test Dataset Structure

In [ ]:
# ===============================================================
# Verify held-out test dataset structure
# ===============================================================

test_structure_rows = []

for dataset, test_file in TEST_FILES.items():

    test_df = pd.read_csv(
        test_file,
        dtype={
            ID_COLUMN: str,
        },
    )

    if ID_COLUMN not in test_df.columns:
        raise ValueError(
            f"{ID_COLUMN!r} missing from {test_file.name}."
        )

    if TARGET not in test_df.columns:
        raise ValueError(
            f"{TARGET!r} missing from {test_file.name}."
        )

    duplicated_ids = (
        test_df[
            ID_COLUMN
        ]
        .duplicated()
        .sum()
    )

    missing_ids = (
        test_df[
            ID_COLUMN
        ]
        .isna()
        .sum()
    )

    missing_targets = (
        test_df[
            TARGET
        ]
        .isna()
        .sum()
    )

    test_structure_rows.append(
        {
            "Dataset":
                dataset,

            "Rows":
                len(test_df),

            "Unique Participants":
                test_df[
                    ID_COLUMN
                ].nunique(),

            "Duplicated Participants":
                duplicated_ids,

            "Missing Participant IDs":
                missing_ids,

            "Missing Targets":
                missing_targets,

            "Total Columns":
                test_df.shape[1],
        }
    )


test_structure_check = pd.DataFrame(
    test_structure_rows
)

display(test_structure_check)

In [ ]:
assert (
    test_structure_check[
        "Duplicated Participants"
    ]
    == 0
).all(), (
    "Duplicated participants detected "
    "in one or more test datasets."
)

assert (
    test_structure_check[
        "Missing Participant IDs"
    ]
    == 0
).all(), (
    "Missing participant IDs detected."
)

assert (
    test_structure_check[
        "Missing Targets"
    ]
    == 0
).all(), (
    "Missing target labels detected."
)

print(
    "Held-out Test datasets verified"
)

#### 5. Evaluate Selected Models on Held-Out Test Data

In [ ]:
# =============================================================================
# Allowed missing feature for Week 5 pipeline compatibility
# =============================================================================

ALLOWED_MISSING_FEATURES = {"urinary_count",}

In [ ]:
# =============================================================================
# Evaluate validation-selected models on held-out test data
# =============================================================================

test_results_rows = []
test_prediction_rows = []
test_probability_rows = []

classification_report_store = {}
confusion_matrix_store = {}

for _, candidate in selected_candidates.iterrows():

    dataset = candidate["dataset"]
    model_name = candidate["model"]

    print("\n" + "=" * 80)
    print(f"{dataset} | {model_name}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Load fitted Week 5 pipeline
    # -------------------------------------------------------------------------

    model_path = MODEL_FILES[
        (
            dataset,
            model_name,
        )
    ]

    pipeline = joblib.load(
        model_path
    )

    # -------------------------------------------------------------------------
    # Load held-out test dataset
    # -------------------------------------------------------------------------

    test_file = TEST_FILES[
        dataset
    ]

    test_df = pd.read_csv(
        test_file,
        dtype={
            ID_COLUMN: str,
        },
    )

    y_test = (
        test_df[
            TARGET
        ]
        .copy()
    )

    # -------------------------------------------------------------------------
    # Recover the exact feature schema used in Week 5
    # -------------------------------------------------------------------------

    if hasattr(
        pipeline,
        "feature_names_in_",
    ):

        expected_columns = list(
            pipeline.feature_names_in_
        )

    else:

        expected_columns = [
            column
            for column in test_df.columns
            if column not in {
                TARGET,
                ID_COLUMN,
            }
        ]

    # -------------------------------------------------------------------------
    # Check missing features
    # -------------------------------------------------------------------------

    missing_columns = [
        column
        for column in expected_columns
        if column not in test_df.columns
    ]

    unsupported_missing = [
        column
        for column in missing_columns
        if column not in ALLOWED_MISSING_FEATURES
    ]

    if unsupported_missing:

        raise ValueError(
            "Test data are missing "
            f"{len(unsupported_missing)} "
            "unsupported feature(s): "
            f"{unsupported_missing[:10]}"
        )

    added_missing = []

    for column in missing_columns:

        test_df[
            column
        ] = np.nan

        added_missing.append(
            column
        )

    # -------------------------------------------------------------------------
    # Align test predictors to fitted pipeline
    # -------------------------------------------------------------------------

    X_test = (
        test_df[
            expected_columns
        ]
        .copy()
    )

    print(
        "Model expects:",
        len(expected_columns),
        "raw features"
    )

    print(
        "Test aligned:",
        X_test.shape[1],
        "raw features"
    )

    if added_missing:

        print(
            "Compatibility handling:",
            added_missing,
            "supplied as NaN",
        )

    # -------------------------------------------------------------------------
    # Final test prediction only
    # NO fit(), retuning, or feature-selection changes
    # -------------------------------------------------------------------------

    y_pred = pipeline.predict(
        X_test
    )

    # -------------------------------------------------------------------------
    # Shared evaluation utilities
    # -------------------------------------------------------------------------

    metrics = calculate_metrics(
        y_test,
        y_pred,
    )

    classification_report_df = (
        get_classification_report(
            y_test,
            y_pred,
        )
    )

    confusion_df = (
        get_confusion_matrix(
            y_test,
            y_pred,
        )
    )

    # -------------------------------------------------------------------------
    # Store model-level metrics
    # -------------------------------------------------------------------------

    test_results_rows.append(
        {
            "dataset":
                dataset,

            "model":
                model_name,

            "artifact_filename":
                model_path.name,

            "test_file":
                test_file.name,

            "n_test":
                len(y_test),

            "added_missing_features":
                ",".join(
                    added_missing
                )
                if added_missing
                else "",

            **metrics,
        }
    )

    result_key = (
        f"{dataset}__{model_name}"
    )

    classification_report_store[
        result_key
    ] = classification_report_df

    confusion_matrix_store[
        result_key
    ] = confusion_df

    # -------------------------------------------------------------------------
    # Store participant-level predictions
    # -------------------------------------------------------------------------

    participant_ids = (
        test_df[
            ID_COLUMN
        ]
        .astype(str)
        .values
    )

    for (
        participant_id,
        true_label,
        predicted_label,
    ) in zip(
        participant_ids,
        y_test,
        y_pred,
    ):

        test_prediction_rows.append(
            {
                ID_COLUMN:
                    participant_id,

                "dataset":
                    dataset,

                "model":
                    model_name,

                "true_label":
                    true_label,

                "predicted_label":
                    predicted_label,
            }
        )

    # -------------------------------------------------------------------------
    # Store predicted probabilities
    # -------------------------------------------------------------------------

    if hasattr(
        pipeline,
        "predict_proba",
    ):

        probabilities = (
            pipeline.predict_proba(
                X_test
            )
        )

        classes = getattr(
            pipeline,
            "classes_",
            None,
        )

        if (
            classes is None
            and hasattr(
                pipeline,
                "named_steps",
            )
        ):

            classifier = (
                pipeline
                .named_steps
                .get(
                    "classifier"
                )
            )

            classes = getattr(
                classifier,
                "classes_",
                np.arange(
                    probabilities.shape[1]
                ),
            )

        if classes is None:

            classes = np.arange(
                probabilities.shape[1]
            )

        for row_index, (
            participant_id,
            true_label,
        ) in enumerate(
            zip(
                participant_ids,
                y_test,
            )
        ):

            probability_row = {
                ID_COLUMN:
                    participant_id,

                "dataset":
                    dataset,

                "model":
                    model_name,

                "true_label":
                    true_label,
            }

            for (
                class_index,
                class_label,
            ) in enumerate(
                classes
            ):

                probability_row[
                    f"prob_class_{class_label}"
                ] = probabilities[
                    row_index,
                    class_index,
                ]

            test_probability_rows.append(
                probability_row
            )

    print(
        f"✓ Final test successful | "
        f"Macro F1={metrics['macro_f1']:.4f}"
    )

#### 6. Final Test Results

The final performance metrics, participant-level predictions, and predicted class probabilities are consolidated and saved for the three validation-selected candidate models.

In [ ]:
# =============================================================================
# Build final test result tables
# =============================================================================

test_results = pd.DataFrame(
    test_results_rows
)

test_predictions = pd.DataFrame(
    test_prediction_rows
)

test_probabilities = pd.DataFrame(
    test_probability_rows
)

display(
    test_results[
        [
            "dataset",
            "model",
            "n_test",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ].round(4)
)

#### Observation: 

The three validation-selected models were successfully evaluated on the held-out test dataset (71 participants). 

Among them, the Full Multimodal Random Forest achieved the strongest overall performance, with the highest Macro F1-score (0.673), balanced accuracy (0.709), and precision (0.654). The Wearable + Questionnaire Random Forest obtained the same overall accuracy (0.704) but a slightly lower Macro F1-score (0.654), indicating slightly less balanced performance across the three diagnostic classes. The Demographics + Questionnaire Logistic Regression showed the lowest performance, with a Macro F1-score of 0.586 and balanced accuracy of 0.617.

This means the results show that the model using all available data (Full Multimodal) performed best when predicting the three diagnostic groups.

In [ ]:
# =============================================================================
# Final test output paths
# =============================================================================

TEST_RESULTS_OUTPUT = (
    METRICS_DIR
    / "test_results.csv"
)

TEST_PREDICTIONS_OUTPUT = (
    METRICS_DIR
    / "test_predictions.csv"
)

TEST_PROBABILITIES_OUTPUT = (
    METRICS_DIR
    / "test_probabilities.csv"
)

In [ ]:
# =============================================================================
# Save final test results
# =============================================================================

test_results.to_csv(
    TEST_RESULTS_OUTPUT,
    index=False,
)

test_predictions.to_csv(
    TEST_PREDICTIONS_OUTPUT,
    index=False,
)

if not test_probabilities.empty:

    test_probabilities.to_csv(
        TEST_PROBABILITIES_OUTPUT,
        index=False,
    )

print(
    "Saved:",
    TEST_RESULTS_OUTPUT,
)

print(
    "Saved:",
    TEST_PREDICTIONS_OUTPUT,
)

if not test_probabilities.empty:

    print(
        "Saved:",
        TEST_PROBABILITIES_OUTPUT,
    )

#### 7. Final Classification Reports and Confusion Matrices

In [ ]:
# ====================================================
# Create safe filenames
# ====================================================

def clean_text(value):
    """Convert text to a safe lowercase filename component."""

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value)
        .strip()
        .lower(),
    ).strip("_")

In [ ]:
# =============================================================================
# Save classification reports and confusion matrices
# =============================================================================

saved_classification_reports = []
saved_confusion_matrices = []

for result_key in classification_report_store:

    dataset, model_name = (
        result_key.split(
            "__",
            maxsplit=1,
        )
    )

    safe_name = (
        f"{clean_text(dataset)}"
        f"_{clean_text(model_name)}"
    )

    # ---------------------------------------------------
    # Classification report
    # ---------------------------------------------------

    classification_report_df = (
        classification_report_store[
            result_key
        ]
    )

    report_path = (
        METRICS_DIR
        / (
            f"{safe_name}"
            "_test_classification_report.csv"
        )
    )

    classification_report_df.to_csv(
        report_path,
        index=True,
    )

    saved_classification_reports.append(
        report_path
    )

    # -----------------------------------------------
    # Confusion matrix
    # -----------------------------------------------

    confusion_df = (
        confusion_matrix_store[
            result_key
        ]
    )

    print(
        "\n",
        dataset,
        "|",
        model_name,
    )

    display(
        confusion_df
    )

    confusion_path = (
        METRICS_DIR
        / (
            f"{safe_name}"
            "_test_confusion_matrix.csv"
        )
    )

    confusion_df.to_csv(
        confusion_path,
        index=True,
    )

    saved_confusion_matrices.append(
        confusion_path
    )


print(
    "\nClassification reports saved:",
    len(saved_classification_reports),
)

print(
    "Confusion matrices saved:",
    len(saved_confusion_matrices),
)

#### Observation: 

The confusion matrices show that all three models classified Parkinson’s Disease (PD) more successfully than the Other Movement Disorder class. 

The Full Multimodal Random Forest correctly classified 11 of 12 Healthy participants, 31 of 42 PD participants, and 8 of 17 Other participants. 

The Wearable + Questionnaire Random Forest correctly identified the largest number of PD participants (33 of 42), while the Demographics + Questionnaire Logistic Regression showed more classification errors across the three groups. 

Overall, the main difficulty for all models was distinguishing participants in the Other class, which was frequently predicted as PD.

#### 8. Validation vs. Test Performance

The final test performance of each validation-selected candidate is compared with its corresponding validation performance.

In [ ]:
# ===========================================================
# Build validation vs. test comparison
# ===========================================================

comparison_rows = []

for _, candidate in selected_candidates.iterrows():

    dataset = candidate[
        "dataset"
    ]

    model_name = candidate[
        "model"
    ]

    validation_row = (
        validation_results[
            (
                validation_results[
                    "dataset"
                ]
                == dataset
            )
            & (
                validation_results[
                    "model"
                ]
                == model_name
            )
        ]
        .iloc[0]
    )

    test_row = (
        test_results[
            (
                test_results[
                    "dataset"
                ]
                == dataset
            )
            & (
                test_results[
                    "model"
                ]
                == model_name
            )
        ]
        .iloc[0]
    )

    comparison_rows.append(
        {
            "dataset":
                dataset,

            "model":
                model_name,

            "validation_accuracy":
                validation_row[
                    "accuracy"
                ],

            "test_accuracy":
                test_row[
                    "accuracy"
                ],

            "validation_balanced_accuracy":
                validation_row[
                    "balanced_accuracy"
                ],

            "test_balanced_accuracy":
                test_row[
                    "balanced_accuracy"
                ],

            "validation_macro_f1":
                validation_row[
                    "macro_f1"
                ],

            "test_macro_f1":
                test_row[
                    "macro_f1"
                ],

            "validation_precision_macro":
                validation_row[
                    "precision_macro"
                ],

            "test_precision_macro":
                test_row[
                    "precision_macro"
                ],

            "validation_recall_macro":
                validation_row[
                    "recall_macro"
                ],

            "test_recall_macro":
                test_row[
                    "recall_macro"
                ],
        }
    )

validation_test_comparison = pd.DataFrame(
    comparison_rows
)

In [ ]:
# =========================================================
# Calculate validation-to-test changes
# =========================================================

validation_test_comparison[
    "accuracy_change"
] = (
    validation_test_comparison[
        "test_accuracy"
    ]
    - validation_test_comparison[
        "validation_accuracy"
    ]
)

validation_test_comparison[
    "balanced_accuracy_change"
] = (
    validation_test_comparison[
        "test_balanced_accuracy"
    ]
    - validation_test_comparison[
        "validation_balanced_accuracy"
    ]
)

validation_test_comparison[
    "macro_f1_change"
] = (
    validation_test_comparison[
        "test_macro_f1"
    ]
    - validation_test_comparison[
        "validation_macro_f1"
    ]
)

In [ ]:
display(
    validation_test_comparison[
        [
            "dataset",
            "model",

            "validation_macro_f1",
            "test_macro_f1",
            "macro_f1_change",

            "validation_balanced_accuracy",
            "test_balanced_accuracy",
            "balanced_accuracy_change",

            "validation_accuracy",
            "test_accuracy",
            "accuracy_change",
        ]
    ].round(4)
)

In [ ]:
# =============================================================================
# Save validation vs. test comparison
# =============================================================================

VALIDATION_TEST_COMPARISON_OUTPUT = (
    TABLES_DIR
    / "validation_test_comparison.csv"
)

validation_test_comparison.to_csv(
    VALIDATION_TEST_COMPARISON_OUTPUT,
    index=False,
)

print(
    "Saved:",
    VALIDATION_TEST_COMPARISON_OUTPUT,
)

#### 9. Final Performance Summary

The held-out test results are summarized for the three models previously selected during validation.

In [ ]:
# =============================================================================
# Final performance summary
# =============================================================================

final_performance_summary = (
    test_results[
        [
            "dataset",
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ]
    .copy()
)

display(
    final_performance_summary.round(4)
)

In [ ]:
display(
    validation_test_comparison[
        [
            "dataset",
            "model",
            "macro_f1_change",
            "balanced_accuracy_change",
            "accuracy_change",
        ]
    ].round(4)
)

In [ ]:
print(
    "Final held-out test evaluation completed "
    "for all validation-selected candidates."
)

print(
    "No fitting, retuning, feature selection, "
    "or test-based model selection was performed."
)

#### 10. Deliverables Check

In [ ]:
# ======================================================
# Verify final test deliverables
# ======================================================

deliverables = [
    {
        "Deliverable":
            "Final test results",

        "Path":
            TEST_RESULTS_OUTPUT,
    },

    {
        "Deliverable":
            "Final test predictions",

        "Path":
            TEST_PREDICTIONS_OUTPUT,
    },

    {
        "Deliverable":
            "Validation vs. test comparison",

        "Path":
            VALIDATION_TEST_COMPARISON_OUTPUT,
    },
]

if not test_probabilities.empty:

    deliverables.append(
        {
            "Deliverable":
                "Final test probabilities",

            "Path":
                TEST_PROBABILITIES_OUTPUT,
        }
    )


deliverables_df = pd.DataFrame(
    deliverables
)

deliverables_df[
    "Status"
] = (
    deliverables_df[
        "Path"
    ]
    .apply(
        lambda path: (
            "READY"
            if (
                Path(path).exists()
                and Path(path).stat().st_size > 0
            )
            else "MISSING"
        )
    )
)

display(
    deliverables_df
)